# GWM Self-Attention Node Classification - Cora Dataset (Colab)

This notebook trains the GWM model with self-attention mechanism for node classification on Google Colab.

**Requirements:**
- GPU: T4, V100, or A100 (16GB+)
- Training time: ~8-12 hours
- Runtime: GPU with High-RAM

**Architecture:**
- Self-attention across multi-hop neighborhoods
- LLaMA-3.2-3B as decoder
- Only trains the self-attention projector

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Setup and Installation

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install dependencies
!pip install -q transformers==4.46.2 torch==2.1.0 accelerate==0.34.0 bitsandbytes==0.45.0
!pip install -q datasets sentencepiece protobuf

## 3. Configuration

Upload your Cora processed data files to `/content/`:
- `cora_train_node_data.jsonl`
- `cora_val_node_data.jsonl`
- `cora_test_node_data.jsonl`
- `train_node_embeddings.pt`
- `val_node_embeddings.pt`
- `test_node_embeddings.pt`

Also upload the training code files:
- `model.py`
- `dataset.py`
- `utils.py`
- `train.py`

In [ ]:
import os

# Paths
DATA_DIR = '/content'  # Data files location
PROJECT_ROOT = '/content/drive/MyDrive/NLP_Research/graph-world-models'
OUTPUT_DIR = f'{PROJECT_ROOT}/trained/GWM/node-classification/self-attn/Cora'

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print("\n✓ Configuration complete")

## 4. HuggingFace Authentication

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Get HF token from Colab secrets
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print("✓ Logged in to HuggingFace")

## 5. Training Configuration

In [ ]:
# Training hyperparameters
LLAMA_MODEL = "meta-llama/Llama-3.2-3B-Instruct"
DATASET_NAME = "cora"

# Model architecture
GRAPH_EMBEDDING_DIM = 2048  # Per-hop embedding dimension
PROJECTOR_HIDDEN_DIM = 4096
NUM_HOPS = 5
NUM_ATTENTION_HEADS = 8
NUM_ATTENTION_LAYERS = 2
DROPOUT = 0.1

# Training parameters
NUM_EPOCHS = 20
BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
LEARNING_RATE = 3e-5
WEIGHT_DECAY = 0.1
WARMUP_STEPS = 50
MAX_GRAD_NORM = 1.0
EARLY_STOPPING_PATIENCE = 5

# Generation parameters
MAX_NEW_TOKENS = 10
TEMPERATURE = 0.1

SEED = 42

## 6. Run Training

In [ ]:
# Build training command
train_cmd = f"""
python train.py \
    --data_dir {DATA_DIR} \
    --dataset_name {DATASET_NAME} \
    --output_dir {OUTPUT_DIR} \
    --llama_model {LLAMA_MODEL} \
    --graph_embedding_dim {GRAPH_EMBEDDING_DIM} \
    --projector_hidden_dim {PROJECTOR_HIDDEN_DIM} \
    --num_hops {NUM_HOPS} \
    --num_attention_heads {NUM_ATTENTION_HEADS} \
    --num_attention_layers {NUM_ATTENTION_LAYERS} \
    --dropout {DROPOUT} \
    --num_epochs {NUM_EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --eval_batch_size {EVAL_BATCH_SIZE} \
    --learning_rate {LEARNING_RATE} \
    --weight_decay {WEIGHT_DECAY} \
    --warmup_steps {WARMUP_STEPS} \
    --max_grad_norm {MAX_GRAD_NORM} \
    --early_stopping_patience {EARLY_STOPPING_PATIENCE} \
    --max_new_tokens {MAX_NEW_TOKENS} \
    --temperature {TEMPERATURE} \
    --seed {SEED} \
    --log_interval 10
"""

print("Training command:")
print(train_cmd)
print("\n" + "="*60)
print("Starting training...")
print("="*60 + "\n")

!{train_cmd}

## 7. View Results

In [ ]:
import json
import matplotlib.pyplot as plt

# Load training history
with open(f'{OUTPUT_DIR}/training_history.json', 'r') as f:
    history = json.load(f)

# Load test results
with open(f'{OUTPUT_DIR}/test_results.json', 'r') as f:
    test_results = json.load(f)

print("="*60)
print("TRAINING RESULTS")
print("="*60)
print(f"Best Validation Accuracy: {test_results['best_val_accuracy']:.4f}")
print(f"Test Accuracy: {test_results['test_accuracy']:.4f}")
print(f"Test Macro Accuracy: {test_results['test_macro_accuracy']:.4f}")
print(f"Test Loss: {test_results['test_loss']:.4f}")

print(f"\nPer-class Accuracy:")
for cls, acc in test_results['class_accuracy'].items():
    print(f"  {cls:20s}: {acc:.4f}")

In [ ]:
# Plot training curves
epochs = [h['epoch'] for h in history]
train_loss = [h['train_loss'] for h in history]
val_loss = [h['val_loss'] for h in history]
val_acc = [h['val_accuracy'] for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Loss curves
ax1.plot(epochs, train_loss, 'b-', label='Train Loss', linewidth=2)
ax1.plot(epochs, val_loss, 'r-', label='Val Loss', linewidth=2)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Accuracy curve
ax2.plot(epochs, val_acc, 'g-', label='Val Accuracy', linewidth=2)
ax2.axhline(y=test_results['test_accuracy'], color='orange', linestyle='--', 
            label=f"Test Accuracy: {test_results['test_accuracy']:.4f}", linewidth=2)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy', fontsize=12)
ax2.set_title('Validation Accuracy', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Saved training curves to {OUTPUT_DIR}/training_curves.png")

## 8. Sample Predictions

In [ ]:
# Load and display sample predictions
with open(f'{OUTPUT_DIR}/test_predictions.json', 'r') as f:
    predictions = json.load(f)

print("Sample Predictions (first 10):")
print("="*80)

for i, pred in enumerate(predictions[:10]):
    status = "✓" if pred['prediction'] == pred['label'] else "✗"
    print(f"{status} Node {pred['node_id']}:")
    print(f"   Predicted: {pred['prediction']}")
    print(f"   Label: {pred['label']}")
    print(f"   Generated: {pred['generated_text'][:100]}...")
    print()

## 9. Run Inference (Optional)

Run inference on a specific checkpoint.

In [ ]:
# Run inference using the best checkpoint
inference_cmd = f"""
python inference.py \
    --checkpoint {OUTPUT_DIR}/checkpoint_best.pt \
    --data_dir {DATA_DIR} \
    --dataset_name {DATASET_NAME} \
    --split test \
    --llama_model {LLAMA_MODEL} \
    --graph_embedding_dim {GRAPH_EMBEDDING_DIM} \
    --projector_hidden_dim {PROJECTOR_HIDDEN_DIM} \
    --num_hops {NUM_HOPS} \
    --num_attention_heads {NUM_ATTENTION_HEADS} \
    --num_attention_layers {NUM_ATTENTION_LAYERS} \
    --batch_size {EVAL_BATCH_SIZE} \
    --max_new_tokens {MAX_NEW_TOKENS} \
    --temperature {TEMPERATURE} \
    --output_file {OUTPUT_DIR}/inference_predictions.json
"""

!{inference_cmd}